# Test Model — Load from HF Hub + test on image/video

Downloads a trained ViT gender checkpoint from HuggingFace Hub, then:
1. Test on a single image — upload image, see prediction + confidence
2. Test on a video — upload video, get annotated MP4 with colored boxes

**No training in this notebook.** Just inference.


## 0. Setup

In [ ]:
!pip install -q opencv-python-headless ultralytics transformers huggingface_hub

import torch, os, json, time, shutil
from pathlib import Path
import numpy as np
from PIL import Image
import cv2

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())
print('Using device:', DEVICE)


## 1. Config

In [ ]:
# Edit this to point at your HF repo (stage 1 or stage 2)
HF_REPO = 'abhshkp/footfall-analysis-vit-stage1'
# Or stage 2: 'abhshkp/footfall-gender-vit-retail-finetuned'

CKPT_DIR = '/content/checkpoint'
print('Will load from:', HF_REPO)


## 2. Download checkpoint from HuggingFace Hub

In [ ]:
from huggingface_hub import snapshot_download
print(f'Downloading checkpoint from {HF_REPO}...')
snapshot_download(
    repo_id=HF_REPO,
    local_dir=CKPT_DIR,
    repo_type='model'
)
!ls -lh {CKPT_DIR}
with open(f'{CKPT_DIR}/config.json') as f:
    cfg = json.load(f)
print(f'\nBackbone: {cfg["vit_model_name"]}')
print(f'Labels:   {cfg["id2label"]}')


## 3. Load model

In [ ]:
import torch.nn as nn
from transformers import ViTModel

class GenderClassifier(nn.Module):
    def __init__(self, vit_model_name='google/vit-base-patch16-224', num_labels=2):
        super().__init__()
        self.vit = ViTModel.from_pretrained(vit_model_name)
        self.head = nn.Linear(self.vit.config.hidden_size, num_labels)
        self.num_labels = num_labels
        self.id2label = {0: 'female', 1: 'male'}
    def forward(self, pixel_values):
        out = self.vit(pixel_values=pixel_values)
        return self.head(out.pooler_output)
    @classmethod
    def load(cls, ckpt_dir, device='cpu'):
        with open(Path(ckpt_dir) / 'config.json') as f: cfg = json.load(f)
        m = cls(cfg['vit_model_name'], cfg.get('num_labels', 2))
        state = torch.load(Path(ckpt_dir) / 'pytorch_model.bin', map_location=device)
        m.load_state_dict(state, strict=False)
        m.to(device).eval()
        return m

from transformers import ViTImageProcessor
_PROC = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')
def val_transform(pil_img):
    return _PROC(images=pil_img, return_tensors='pt')['pixel_values'][0]

model = GenderClassifier.load(CKPT_DIR, device=DEVICE)
print('Model loaded.')


## 4a. Test on a single image

In [ ]:
from google.colab import files
from IPython.display import display
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO

# Try to initialize person detector
try:
    detector = YOLO('yolov8n.pt')
except Exception as e:
    print('Could not load YOLO detector, falling back to full-image crop. Error:', e)
    detector = None

print('Upload a test image (face or full body)')
up = files.upload()
img_path = list(up.keys())[0]

img = Image.open(img_path).convert('RGB')
w, h = img.size

# 1. Run Detection & Define Bounding Box
bbox = [0, 0, w, h]  # default fallback to whole image
if detector is not None:
    det_results = detector(img, classes=[0], conf=0.3, verbose=False)[0]
    if len(det_results.boxes) > 0:
        # Crop around the first detected person
        box = det_results.boxes[0]
        bbox = list(map(int, box.xyxy[0].tolist()))

# Panel 1 Frame with red Bounding Box
frame_with_box = img.copy()
draw = ImageDraw.Draw(frame_with_box)
draw.rectangle(bbox, outline=(255, 0, 0), width=max(2, int(min(w, h) * 0.01)))

# 2. Crop the Person
crop_img = img.crop(bbox)

# 3. Preprocess and Draw 16x16 Patch Grid on a 224x224 Copy
resized_img = crop_img.resize((224, 224), Image.Resampling.BILINEAR)
patchified_img = resized_img.copy()
draw_patches = ImageDraw.Draw(patchified_img)
grid_size = 16
for i in range(grid_size, 224, grid_size):
    draw_patches.line([(i, 0), (i, 224)], fill=(255, 255, 255), width=1)
    draw_patches.line([(0, i), (224, i)], fill=(255, 255, 255), width=1)

# Run classification inference on the crop
pixel_values = val_transform(crop_img).unsqueeze(0).to(DEVICE)
with torch.no_grad():
    logits = model(pixel_values)
    probs = torch.softmax(logits, dim=-1)[0]
pred = int(probs.argmax())
pred_label = model.id2label[pred]

# 4. Plot prediction probabilities
probs_list = [probs[1].item(), probs[0].item()]  # [Male, Female]
classes = ['Male', 'Female']
colors = ['#3182bd', '#de2d26']

# Generate 4-panel visual figure
fig, axes = plt.subplots(1, 4, figsize=(20, 5), dpi=300)

axes[0].imshow(frame_with_box)
axes[0].set_title('1. Input Frame (with Detection)', fontsize=14, fontweight='bold', pad=10)
axes[0].axis('off')

axes[1].imshow(crop_img)
axes[1].set_title('2. Cropped Pedestrian', fontsize=14, fontweight='bold', pad=10)
axes[1].axis('off')

axes[2].imshow(patchified_img)
axes[2].set_title('3. Preprocessed & Patchified\n(224x224 Grid)', fontsize=14, fontweight='bold', pad=10)
axes[2].axis('off')

bars = axes[3].barh(classes, probs_list, color=colors, height=0.4)
axes[3].set_xlim(0, 1.1)
axes[3].set_title(f'4. Prediction: {pred_label.upper()}', fontsize=14, fontweight='bold', pad=10)
axes[3].set_xlabel('Probability Score', fontsize=12)
axes[3].spines['top'].set_visible(False)
axes[3].spines['right'].set_visible(False)
axes[3].spines['left'].set_color('#cccccc')
axes[3].spines['bottom'].set_color('#cccccc')

for bar in bars:
    width = bar.get_width()
    axes[3].text(width + 0.02, bar.get_y() + bar.get_height()/2,
                 f'{width*100:.1f}%',
                 ha='left', va='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\nPrediction: {pred_label}')
print(f'  female: {probs[0].item():.3f}')
print(f'  male:   {probs[1].item():.3f}')


#### 4b. Test on batch of images

In [ ]:
# === Batch image test ===
# Upload multiple JPGs at once, get a prediction for each.
# Useful for testing edge cases: long-haired men, children, uniforms, etc.
#
# Where to get edge case images:
# - Google Images: "man with long hair", "woman short hair", "child shopping"
# - Pexels/Unsplash: free stock photos of diverse people
# - Your own CCTV: use the frame extraction cell below
# - Your training data: /content/data/PETA/images/ (if stage 1 data is still loaded)

from google.colab import files
from IPython.display import display, HTML
import pandas as pd

print('Upload multiple test images (JPG/PNG). Select all at once.')
up = files.upload()
image_paths = list(up.keys())
print(f'\nGot {len(image_paths)} images. Running inference...\n')

results = []
for img_path in image_paths:
    img = Image.open(img_path).convert('RGB')
    pixel_values = val_transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(pixel_values)
        probs = torch.softmax(logits, dim=-1)[0]
    pred = int(probs.argmax())
    label = model.id2label[pred]
    conf = probs[pred].item()
    results.append({
        'image': img_path,
        'prediction': label,
        'confidence': f'{conf:.3f}',
        'male_prob': f'{probs[1].item():.3f}',
        'female_prob': f'{probs[0].item():.3f}',
    })

# Show results table
df = pd.DataFrame(results)
print('\n=== Results ===')
print(df.to_string(index=False))

# Show each image with its prediction
print('\n=== Images with predictions ===')
for r in results:
    img = Image.open(r['image']).convert('RGB')
    print(f"\n{r['image']}: {r['prediction']} (conf={r['confidence']})")
    display(img.resize((200, 200)))

# Summary
print(f'\n=== Summary ===')
print(f'Total: {len(results)} images')
n_male = sum(1 for r in results if r['prediction'] == 'male')
n_female = sum(1 for r in results if r['prediction'] == 'female')
print(f'  male:   {n_male}')
print(f'  female: {n_female}')
n_low_conf = sum(1 for r in results if float(r['confidence']) < 0.7)
print(f'  low confidence (<0.7): {n_low_conf}')

## 5. Test on a video

Upload a video, get an annotated MP4 with colored gender boxes.
- Blue box = male
- Pink box = female

Uses YOLOv8n for person detection + your ViT for gender.


In [ ]:
from google.colab import files
from ultralytics import YOLO

print('Upload a video file (.mp4, .mov, etc.)')
up = files.upload()
video_path = list(up.keys())[0]
print(f'Using video: {video_path}')

detector = YOLO('yolov8n.pt')  # auto-downloads

CONF_THRESH = 0.4
FRAME_SKIP = 1
OUTPUT_PATH = '/content/output_annotated.mp4'

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 25
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'Video: {w}x{h} @ {fps:.1f}fps, {n_frames} frames')

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (w, h))

COLORS = {'male': (255, 140, 0), 'female': (255, 20, 147)}  # BGR

frame_idx = 0
t0 = time.time()
while True:
    ret, frame = cap.read()
    if not ret: break
    if frame_idx % FRAME_SKIP == 0:
        results = detector(frame, classes=[0], conf=CONF_THRESH, verbose=False)[0]
        for box in results.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)
            if x2 <= x1 or y2 <= y1: continue
            crop = frame[y1:y2, x1:x2]
            pil = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
            pv = val_transform(pil).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                logits = model(pv)
                probs = torch.softmax(logits, dim=-1)[0]
            pred = int(probs.argmax())
            label = model.id2label[pred]
            conf = probs[pred].item()
            color = COLORS[label]
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            text = f'{label} {conf:.2f}'
            (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            cv2.rectangle(frame, (x1, y1-th-8), (x1+tw+4, y1), color, -1)
            cv2.putText(frame, text, (x1+2, y1-4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
    writer.write(frame)
    frame_idx += 1
    if frame_idx % 50 == 0:
        elapsed = time.time() - t0
        eta = elapsed / frame_idx * (n_frames - frame_idx)
        print(f'  frame {frame_idx}/{n_frames}  elapsed={elapsed:.0f}s  eta={eta:.0f}s')

cap.release()
writer.release()
print(f'\nDone. Annotated video saved to: {OUTPUT_PATH}')


In [ ]:
from google.colab import files
files.download(OUTPUT_PATH)


## Done

Annotated video downloaded. Check:
- Are all people boxed? (detection coverage)
- Are boxes on people, not shelves/mannequins? (false positives)
- Do gender labels match what you see? (classification accuracy)

If accuracy is poor:
- Try stage 2 fine-tuning on your own CCTV data (`train_stage2.ipynb`)
- Lower YOLO conf threshold (CONF_THRESH = 0.3) to catch more people
- Raise FRAME_SKIP to 3 to speed up processing (with slight accuracy loss)
